In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load data (same as Step 2)
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
category_translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

print("✅ Data loaded successfully")
print(f"Orders: {orders.shape[0]} rows")
print(f"Reviews: {reviews.shape[0]} rows")

✅ Data loaded successfully
Orders: 99441 rows
Reviews: 99224 rows


Convert timestamps & create key features

In [2]:
# Convert timestamps to datetime
date_columns = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

# Calculate delivery metrics (key business features)
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['estimated_days'] = (orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']).dt.days
orders['delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
orders['is_late'] = orders['delay_days'] > 0
orders['delay_bucket'] = pd.cut(orders['delay_days'], 
                                bins=[-1, 0, 3, 7, 30, 100], 
                                labels=['On Time', 'Minor Delay', 'Moderate Delay', 'Major Delay', 'Extreme Delay'])

print("✅ Key delivery features created")
print("\nDelay distribution:")
print(orders['delay_bucket'].value_counts())
print("\nLate orders %:", orders['is_late'].mean()*100, "%")

✅ Key delivery features created

Delay distribution:
delay_bucket
Major Delay       2518
Minor Delay       1870
Moderate Delay    1802
On Time           1292
Extreme Delay      306
Name: count, dtype: int64

Late orders %: 6.571736004263835 %


Clean & join tables

In [3]:
# Clean reviews - handle multiple reviews per order
reviews_clean = reviews.groupby('order_id')['review_score'].mean().reset_index()
reviews_clean.rename(columns={'review_score': 'avg_review_score'}, inplace=True)

# Merge core tables
df_core = (orders.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
                 .merge(reviews_clean, on='order_id', how='left')
                 .merge(order_items[['order_id', 'product_id', 'seller_id', 'price', 'freight_value']], 
                        on='order_id', how='left'))

# Category mapping
category_map = dict(zip(category_translation['product_category_name'], 
                       category_translation['product_category_name_english']))
products['category_english'] = products['product_category_name'].map(category_map)

# Final merge
df_final = df_core.merge(products[['product_id', 'category_english']], on='product_id', how='left')

print("✅ Final dataset ready")
print(f"Shape: {df_final.shape}")
print("\nKey columns created:")
print("- delay_days")
print("- is_late") 
print("- delay_bucket")
print("- avg_review_score")
print("- category_english")

✅ Final dataset ready
Shape: (113425, 20)

Key columns created:
- delay_days
- is_late
- delay_bucket
- avg_review_score
- category_english


Save cleaned data

In [4]:
# Save for MySQL import
df_final.to_csv("../data/processed/ecommerce_cleaned.csv", index=False)

# Quick quality check
print("✅ Data saved to data/processed/")
print(f"Final dataset: {df_final.shape}")
print("\nSample:")
print(df_final[['order_id', 'customer_state', 'delay_days', 'is_late', 'delay_bucket', 'avg_review_score', 'category_english']].head())
print("\nMissing values check:")
print(df_final[['delay_days', 'avg_review_score', 'customer_state']].isnull().sum())

✅ Data saved to data/processed/
Final dataset: (113425, 20)

Sample:
                           order_id customer_state  delay_days  is_late  \
0  e481f51cbdc54678b7cc49136f2d6af7             SP        -8.0    False   
1  53cdb2fc8bc7dce0b6741e2150273451             BA        -6.0    False   
2  47770eb9100c2d0c44946d9cf07ec65d             GO       -18.0    False   
3  949d5b44dbf5de918fe9c16f97b45f8a             RN       -13.0    False   
4  ad21c59c0840e6cb83a9ceb5573f8159             SP       -10.0    False   

  delay_bucket  avg_review_score category_english  
0          NaN               4.0       housewares  
1          NaN               4.0        perfumery  
2          NaN               5.0             auto  
3          NaN               5.0         pet_shop  
4          NaN               5.0       stationery  

Missing values check:
delay_days          3229
avg_review_score     961
customer_state         0
dtype: int64
